# SNN Framework Benchmark — Colab Runner (DSEC, optical-flow regression)

Runs the 4 regression framework variants (Norse, snnTorch, SpikingJelly, Sinabs — `learning/frameworks/personal/`) on DSEC, full-scale: 10 epochs, over a **size-budgeted** subset of DSEC's recordings (~1GB for train, ~500MB for test — DSEC's full 18-recording optical-flow split is far larger than a fresh Colab disk needs for this comparison).

**Train-only, by design** — see `docs/Haseeb-open-items.md`: `SNNTrainer` is verified regression-aware (dense flow-target loss, no TRADES, no argmax accuracy), but `SNNTester`/`AdversarialEvaluator` still assume classification-shaped output (confusion matrix, argmax predictions) and are documented to crash on a regression run. `main.py` prints the same caveat. This notebook only trains and records the loss curve — it does not attempt the test/adversarial phases `run_on_colab.ipynb`'s other datasets run.

Because this doesn't go through `run_benchmark.py` (that script has no DSEC size-budget option), the pipeline code lives directly in this notebook's cells below instead of shelling out to it.

No Google Drive mounting here — everything stays on this Colab session's local disk (`/content/...`). Download results (last cell) before closing the tab.

Runtime -> Change runtime type -> GPU, before running anything below.

## 1. Get the codebase

Clones from the repo referenced in this project's own docs (`docs/roadmap.md`). If your remote/branch differs, edit the URL/branch below before running.

In [ ]:
REPO_URL = "https://github.com/Zuzu3290/SNNs-auf-GPUs.git"
BRANCH = "46-cache_engine"  # the branch this benchmark work is actually on -- main is 7 commits behind and doesn't have it

!git clone --branch {BRANCH} {REPO_URL} /content/SNNs-auf-GPUs
%cd /content/SNNs-auf-GPUs

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

import torch
print("torch:", torch.__version__, " CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected -- check Runtime > Change runtime type > GPU")

## 3. Size-budgeted DSEC dataloaders

DSEC's normal loading path (`event_data_workflow/data_pipeline.py`'s `load_raw()`) downloads all 18 optical-flow recordings, then splits 80/20 across the combined samples -- too much for a fresh Colab disk here. `BudgetedDSECEncoder` below overrides just the raw-loading step: it downloads DSEC recordings by name, one at a time, stopping as soon as cumulative on-disk size crosses budget -- ~1GB for train, ~500MB for test, two disjoint recording sets. Everything past that (caching, framing, DataLoader construction) is unchanged, inherited from `NeuromorphicEncoder`.

A single DSEC recording can itself exceed the budget, so this is a best-effort stopping point, not an exact byte cap.

In [ ]:
import sys, json, time, traceback
from pathlib import Path

ROOT = Path("/content/SNNs-auf-GPUs")

import torch
import tonic
import tonic.transforms as transforms

from skeleton import Settings
from event_data_workflow.data_pipeline import NeuromorphicEncoder, DATA_DIR, logger
from event_data_workflow.regression_datasets import DSECRaw
from learning.training import SNNTrainer
from learning.frameworks.personal import (
    SNN_NORSE_REGRESSION, SNN_TORCH_REGRESSION, SNN_SJ_REGRESSION, SNN_SINABS_REGRESSION,
)

DATA_DIR_OUT = ROOT / "docs" / "results" / "data" / "dsec"
DATA_DIR_OUT.mkdir(parents=True, exist_ok=True)

EPOCHS = 10
TRAIN_BUDGET_BYTES = 1 * 1024 ** 3    # ~1GB
TEST_BUDGET_BYTES = 500 * 1024 ** 2   # ~500MB

MODELS = {
    "norse":  SNN_NORSE_REGRESSION,
    "torch":  SNN_TORCH_REGRESSION,
    "sj":     SNN_SJ_REGRESSION,
    "sinabs": SNN_SINABS_REGRESSION,
}


def _dir_size_bytes(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(f.stat().st_size for f in path.rglob("*") if f.is_file())


def select_recordings_within_budget(save_to, candidate_names, budget_bytes, already_used):
    """Downloads DSEC recordings one at a time, stopping once the cumulative
    on-disk size of the newly-chosen ones crosses budget_bytes. Skips names
    already claimed by a previous call (keeps train/test disjoint). Already-
    downloaded recordings aren't re-fetched -- tonic checks existence itself."""
    dsec_root = Path(save_to) / "DSEC"
    chosen, total = [], 0
    for name in candidate_names:
        if name in already_used:
            continue
        DSECRaw(save_to=save_to, split=[name])
        size = _dir_size_bytes(dsec_root / name)
        chosen.append(name)
        total += size
        print(f"    + {name}  ({size / 1e6:.0f} MB, cumulative {total / 1e6:.0f} MB)")
        if total >= budget_bytes:
            break
    if not chosen:
        raise RuntimeError("No DSEC recordings available to select from.")
    return chosen


class BudgetedDSECEncoder(NeuromorphicEncoder):
    """DSEC-only: recording-level, size-budgeted raw loading instead of the
    normal 'download everything, random_split by sample' path."""

    def load_raw(self):
        entry = self.select_dataset()
        if entry["name"] != "DSEC":
            raise RuntimeError(f"BudgetedDSECEncoder is DSEC-only, got '{entry['name']}'")
        sensor_size = entry["sensor_size"]

        all_names = [n for n, has_flow in tonic.datasets.DSEC.recordings["train"].items() if has_flow]
        print(f"[DSEC] Selecting train recordings (budget {TRAIN_BUDGET_BYTES / 1e9:.2f} GB)...")
        train_names = select_recordings_within_budget(str(DATA_DIR), all_names, TRAIN_BUDGET_BYTES, set())
        print(f"[DSEC] Selecting test recordings (budget {TEST_BUDGET_BYTES / 1e6:.0f} MB, disjoint from train)...")
        test_names = select_recordings_within_budget(str(DATA_DIR), all_names, TEST_BUDGET_BYTES, set(train_names))
        print(f"[DSEC] Train recordings: {train_names}")
        print(f"[DSEC] Test recordings : {test_names}")

        raw_train = DSECRaw(save_to=str(DATA_DIR), split=train_names)
        raw_test = DSECRaw(save_to=str(DATA_DIR), split=test_names)

        self.dataset_label = entry["name"]
        self.task_type = "regression"
        self.cfg.TASK_TYPE = self.task_type
        self.cfg.apply_dataset_shape(
            sensor_h=sensor_size[1], sensor_w=sensor_size[0],
            in_channels=sensor_size[2], num_classes=entry["num_classes"],
        )
        self.sensor_size = sensor_size
        logger.info(f"[PIPELINE] Train: {len(raw_train)} windowed samples ({len(train_names)} recordings)")
        logger.info(f"[PIPELINE] Test : {len(raw_test)} windowed samples ({len(test_names)} recordings)")

        self.validate_first_sample(raw_train, "train")
        self.validate_first_sample(raw_test, "test")

        if self.wf.FRAME_MODE == "n_time_bins":
            to_frame = transforms.ToFrame(sensor_size=sensor_size, n_time_bins=self.wf.N_TIME_BINS)
        else:
            to_frame = transforms.ToFrame(sensor_size=sensor_size, time_window=self.wf.TIME_WINDOW_US)
        frame_tf = transforms.Compose([transforms.Denoise(filter_time=10000), to_frame])
        return raw_train, raw_test, frame_tf


cfg = Settings()
cfg.DATASET_NAME = "DSEC"
cfg.EPOCHS = EPOCHS
cfg.ITERA = 10 ** 9
cfg.TRADES_ENABLED = False
if cfg.DEVICE == "auto":
    cfg.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(cfg.DEVICE)

print("Building DSEC dataloaders (size-budgeted)...")
encoder = BudgetedDSECEncoder(cfg)
train_loader, test_loader = encoder.get_dataloaders()
print(f"Train batches: {len(train_loader)}  Test batches: {len(test_loader)} (test loader built but unused -- see cell 0)")

## 4. Train all 4 regression frameworks (train-only — see note in cell 0)

In [ ]:
results = {}
errors = {}

for name, ModelClass in MODELS.items():
    print("\n" + "=" * 60 + "\n  " + name.upper() + "\n" + "=" * 60)
    cfg.FRAMEWORK = name
    try:
        model = ModelClass(cfg)
        trainer = SNNTrainer(model, train_loader, cfg, device)

        t0 = time.perf_counter()
        train_results = trainer.train(
            checkpoint_dir=str(ROOT / "checkpoints" / f"dsec_{name}"),
            csv_path=str(DATA_DIR_OUT / f"{name}_train.csv"),
        )
        train_time_s = time.perf_counter() - t0

        summary = {
            "framework": name,
            "train_time_s": train_time_s,
            "loss_history": train_results["loss_history"],
            "spike_rate_history": train_results["spike_rate_history"],
            "note": "Test/adversarial phases skipped for DSEC -- see cell 0.",
        }
        with open(DATA_DIR_OUT / f"{name}_summary.json", "w") as f:
            json.dump(summary, f, indent=2)
        results[name] = summary
        print(f"  Final loss: {train_results['loss_history'][-1]:.4f}  ({train_time_s:.1f}s)")

        del model, trainer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as e:
        print(f"  !! {name} FAILED: {e}")
        traceback.print_exc()
        errors[name] = str(e)

with open(DATA_DIR_OUT / "errors.json", "w") as f:
    json.dump(errors, f, indent=2)

print("\nDone. Trained:", list(results.keys()), " Failed:", list(errors.keys()))

## 5. View training loss curves

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
for name, summary in results.items():
    plt.plot(summary["loss_history"], label=name)
plt.xlabel("Training iteration")
plt.ylabel("Loss (flow_masked_mse)")
plt.title("DSEC — Training Loss")
plt.legend()
plt.tight_layout()
plt.savefig(DATA_DIR_OUT / "loss_curves.png", dpi=150)
plt.show()

## 6. Download everything before the session ends

No Drive mounting — zips the results and triggers a browser download instead.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/snn_dsec_results", "zip", "docs/results", "data")
files.download("/content/snn_dsec_results.zip")